In [79]:
import json
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from glob import glob
import os
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import Javascript, display
from IPython.display import HTML, Image
from pathlib import Path

In [80]:
def read_results(path):
    #print(f"\tTrying to read {path}")
    if os.path.exists(path):
        with open(path) as f:
            results = json.load(f)
        return results
    else:
        print(f"Cannot find file {path}")
        return None

lang_map = {"eng": "en",
            "fra": "fr",
            "fin": "fi",
            "zho": "zh",
            "ita": "it",
            "deu": "de",
            "vie": "vi",
            "jpn": "ja",
            "ukr": "uk",
            "ell": "el",
            "hin": "hi",
            "bul": "bg",
            "fas": "fa"}

In [81]:
models = [os.path.basename(m) for m in glob("/flash/project_462000883/models/*")] + ["openai__text-embedding-3-small","openai__text-embedding-3-large", "gemini-embedding-001"]

# two names, one for mteb one for my results
#dataset= "ARCChallenge" #"RTE3-multi" #"ARCChallenge" #"ARCChallenge"#"SummEvalSummarization.v2"#"ARCChallenge" #"WebFAQRetrieval" #" "Tatoeba"
#dataset_= "arcchallenge" #"rte3-multi__contradiction" #"summeval"#"tatoeba" #"arcchallenge" #"web-faq-bitext" "summeval" "tatoeba"


dataset_map ={#"ARCChallenge" : "arcchallenge",
              #"SummEvalSummarization.v2": "summeval",
              #"RTE3-multi": "rte3-multi__contradiction",
              "Tatoeba": "tatoeba",
              #"WebFAQRetrieval": "web-faq-bitext"}
}

In [82]:
def parse_mteb_results(models, dataset):
    if dataset == "RTE3-multi":
        return parse_rte(models, dataset)
    print("Reading MTEB results...")
    # path where the results are
    mteb_results_path=lambda x: [i for i in glob(f"mteb_out/{x}/results/{x}/*/{dataset}.json")]
    results = {}
    for model in models:
        #print(f"In {model}")
        if model == "gemini-embedding-001":
            model_to_read = "google__gemini-embedding-001"
        else:
            model_to_read = model
        mteb_results_path_ = mteb_results_path(model_to_read)
        if mteb_results_path_ == []:
            print(f"\tModel {model} missing results from mteb")
            continue
        if len(mteb_results_path_) != 1:
            # the dataset revision has changed, using the first results
            print(f"\t{len(mteb_results_path_)} evaluation files found for {model}. Selecting first results.")
            mteb_results_path_ = [mteb_results_path_[0]]
        
        d = read_results(mteb_results_path_[0])  # [0] just to remove nesting
        for subsplit in d["scores"]["test"]:
            #print(subsplit)
            if not f'{subsplit["hf_subset"]}' in results.keys():
                results[f'{subsplit["hf_subset"]}'] = {}
            results[f'{subsplit["hf_subset"]}'][f'{model}']= subsplit["main_score"]
    return results

def parse_rte(models, dataset):
    results = {}
    langs_rte = ["en", "fr", "it", "de"]
    mteb_results_path="mteb_out/rte_crawled.json"
    j = read_results(mteb_results_path)
    for m in models:
        m_ = m.replace("__", "/")
        if "gemini" in m_:
            m_ = "google/gemini-embedding-001"
        for l in langs_rte:
            if l not in results.keys():
                results[l] = {}
            #print(f"{m_}_{l}")
            results[l][m] = j[f"{m_}_{l}"]
    return results

            



In [ ]:
def parse_bss_results(models, dataset, splits, prompt=False, shuffled=False, method="ICA", dim=32):
    print("Reading BSS results...")
    ginis = {}
    peaks = {}
    peak_indices = {}
     # unneccessarily complex path
    bss_results_path=lambda x: f"stats{'_with_prompt' if prompt else ''}/{method}_{dim}/{dataset}/{x}.json" if isinstance(x, str) else f"stats{'_with_prompt' if prompt else ''}/{method}_{dim}/{dataset}:{x[1]}/{x[0]}.json"
   

    for m in models:
        if "openai" in m:
            m_ = m.replace("openai__","")
        else:
            m_ = m

        for s in splits:
            if s == "default":
                path = bss_results_path(m_)
            else:
                if dataset == "tatoeba":
                    # remap splits to match hf format
                    s_ = lang_map[s.split("-")[1]]+"-"+lang_map[s.split("-")[0]]
                    path = bss_results_path([m_, s_])
                else:
                    path = bss_results_path([m_, s])
            if not os.path.isfile(path):
                print(f"\tmissing results for {path}")
                continue
            else:
                j = read_results(path)
            if s not in ginis.keys():
                ginis[s] = {}
                peaks[s] = {}
                peak_indices[s] = {}
            ginis[s][m] = (dim-1)/dim - j["gini"]  # see analysis/README.md
            peaks[s][m] = j["peak"]
            peak_indices[s][m] = j["peak_indices"]
        
    return ginis, peaks, peak_indices




In [ ]:
def parse_NN_results(models, dataset, splits, prompt=False):
    print("Reading NN-results...")
    NN_results_path= lambda model, data: f"linearity_results{'_with_prompt' if prompt else ''}/{data}/{model}.json"   #old results in linearity_results_cosine_recall
    
    knns = {}
    r2s = {}
    for m in models:
        if "openai" in m:
            m_ = m.replace("openai__","")
        else:
            m_ = m
        for s in splits:
            if s == "default":
                path = NN_results_path(m_, dataset)
            else:
                if dataset == "tatoeba":
                    # remap splits to match hf format
                    s_ = lang_map[s.split("-")[1]]+"-"+lang_map[s.split("-")[0]]
                    path = NN_results_path(m_, dataset+":"+s_)
                else:
                    path = NN_results_path(m_, dataset+":"+s)
            if not os.path.isfile(path):
                print(f"\tmissing results for {path}")
                continue
            else:
                j = read_results(path)
                #print(j)
            if s not in knns.keys():
                knns[s] = {}
                r2s[s] = {}
            
            knns[s][m] = j["knn_scores"]["10"]
            r2s[s][m] = j["r2_affine"]
           
    return knns, r2s



In [85]:
def color_text(s):
    return f"\x1b[31m{s}\x1b[0m"

def analysis(dataset, df, score, against,split="default", pprint=False):
    collect_results = []
    if isinstance(score, str):
        score = [score]
    for s in score:
        x = df[s].tolist()
        y = df[against].tolist()
        pr = pearsonr(x,y)
        sp = spearmanr(x,y)
        p_value = round(pr.pvalue,5)
        p_value = color_text(p_value) if p_value <= 0.05 else p_value
        sp_value = round(sp.pvalue, 5)
        sp_value = color_text(sp_value) if sp_value <= 0.05 else sp_value
        if pprint:
            print(f"\t Statistic: {s} (support {len(x)})\t\tP-Result: {round(pr.statistic, 3)} (p-value {p_value})  \n\
                \t\t\t\tS-Result: {round(sp.statistic, 3)} (p-value {sp_value})")
        collect_results.append({"dataset": dataset, "split":split, "statistic":s, "support": len(x), "pr": (pr.statistic, pr.pvalue), "sp":(sp.statistic, sp.pvalue)})
    return collect_results
        




def make_table(dataset, mteb_scores, dict_of_results):
    results_knn = []
    results_bss = []
    dfs_collect = {}
    for split in mteb_scores.keys():
        dfs = []
        df_mteb = pd.DataFrame([mteb_scores[split]]).T.rename(columns={0:"mteb_score"})
        for name, result in dict_of_results.items():
            df = pd.DataFrame([result[split]]).T.rename(columns={0:name})
            dfs.append(df)
        df = pd.concat([df_mteb] + dfs,  axis=1)
        df = df.reset_index().rename(columns={"index":"model"})
        df = df.dropna()   # drops all NaN values --> i.e. do not analyse unless all results exist
        r1 = analysis(dataset, df, [i for i in dict_of_results.keys() if i in ["knn", "r2_aff"]], "mteb_score", split=split)
        r2 = analysis(dataset, df, [i for i in dict_of_results.keys() if "gini" in i or "peak" in i], "mteb_score", split=split)
        results_knn.append(r1)
        results_bss.append(r2)
        dfs_collect[split] = df
    return results_knn, results_bss, dfs_collect





In [86]:

def format_corr_value(corr_tuple: tuple[float, float]) -> str:
    """Format a (coefficient, pvalue) tuple into a LaTeX string."""
    coeff, pvalue = corr_tuple

    coeff_str = f"{coeff:.3f}"

    # Significance marker based on p-value
    if pvalue > 0.05:
        sig = ""
    elif pvalue > 0.01:
        sig = r"$^{*}$"
    elif pvalue > 0.001:
        sig = r"$^{\dagger}$"
    else:
        sig = r"$^{\ddagger}$"

    # Bold if |coeff| > 0.8
    if abs(coeff) > 0.8:
        return r"\textbf{" + coeff_str + "}" + sig
    else:
        return coeff_str + sig


def to_latex_rows_old_with_no_prompt(results, statistics_to_include="all", metrics_to_include=["spearman", "spearman_wp"]):
    prev_dataset=None
    rows = []
    prev_dataset = None
    val_map = {"knn": "$k$NN", "r2_aff": "$r2$"}
    for data in results:
        for row in data:
            dataset   = row["dataset"]
            split = row["split"]
            statistic = row["statistic"]
            statistic = row["statistic"]
            if statistics_to_include != "all" and statistic not in statistics_to_include and statistic != statistics_to_include:
                print(f"not including {statistic=}")
                continue
            pearson   = format_corr_value(row["pr"])
            spearman  = format_corr_value(row["sp"])

            # Visual separator between datasets
            #if prev_dataset is not None and dataset+split != prev_dataset:
            #    rows.append(r"\midrule")
            #    print(f'midrule added for {dataset+split} != {prev_dataset}')
            #    prev_dataset=dataset+split

            rows.append(f"{dataset}:{split if split!='default' else ''} & {val_map[statistic]} '&'.join{pearson} & {spearman} \\\\")
            #prev_dataset = dataset+split


    return "\n".join(rows)

def to_latex_rows_with_prompt(results, results_with_prompt, statistics_to_include="all", metrics_to_include=["spearman", "spearman_wp"]):
    prev_dataset=None
    rows = []
    prev_dataset = None
    val_map = {"knn": "$k$NN", "r2_aff": "$r2$"}
    dataset_name_map = {"WebFAQRetrieval": "WebFAQ", "SummEvalSummarization.v2": "SummEval"}
    for data, data_wp in zip(results, results_with_prompt):
        for row, row2 in zip(data, data_wp):
            dataset_name   = row["dataset"]
            split = row["split"]
            statistic = row["statistic"]
            if statistics_to_include != "all" and statistic not in statistics_to_include and statistic != statistics_to_include:
                #print(f"not including {statistic=}")
                continue
            d = {}
            d["pearson"]  = format_corr_value(row["pr"])
            d["spearman"]  = format_corr_value(row["sp"])
            d["pearson_wp"] = format_corr_value(row2["pr"])
            d["spearman_wp"] = format_corr_value(row2["sp"])

            if metrics_to_include == "all":
                metrics_to_include = ["pearson", "spearman", "pearson_wp", "spearman_wp"]
            m = " & ".join(d[v] for v in metrics_to_include)
            rows.append(f"{dataset_name_map.get(dataset_name, dataset_name)}{':'+split if split!='default' else ''}  & {m} & {row['support']}\\\\")   #& {val_map.get(statistic, statistic)}  <- if you want this
            #prev_dataset = dataset_name+split
    return "\n".join(rows)
    

In [87]:
def runner(models, dataset, dataset_):
    mteb_scores = parse_mteb_results(models, dataset)
    ginis32, peaks32, peak_indices32 = parse_bss_results(models, dataset_, splits = [i for i in mteb_scores.keys()], method="ICA", dim=32)
    ginis64, peaks64, peak_indices64 = parse_bss_results(models, dataset_, splits = [i for i in mteb_scores.keys()], method="ICA", dim=64)
    #ginis128, peaks128, peak_indices128 = parse_bss_results(models, dataset_, splits = [i for i in mteb_scores.keys()], method="ICA", dim=128)
    knns, r2s = parse_NN_results(models, dataset_, splits = [i for i in mteb_scores.keys()])

    ginis32_p, peaks32_p, peak_indices32_p = parse_bss_results(models, dataset_, splits = [i for i in mteb_scores.keys()], method="ICA", dim=32, prompt=True)
    ginis64_p, peaks64_p, peak_indices64_p = parse_bss_results(models, dataset_, splits = [i for i in mteb_scores.keys()], method="ICA", dim=64, prompt=True)
    #ginis128_p, peaks128_p, peak_indices128_p = parse_bss_results(models, dataset_, splits = [i for i in mteb_scores.keys()], method="ICA", dim=128, prompt=True)
    knns_p, r2s_p = parse_NN_results(models, dataset_, splits = [i for i in mteb_scores.keys()], prompt=True)

    collected_results = {"ginis32": ginis32,  
                         "peaks32": peaks32,
                         "ginis64": ginis64, 
                         "peaks64": peaks64,
                         #"ginis128": ginis128,
                         #"peaks128": peaks128,
                         "knn": knns, 
                         "r2_aff": r2s}
    collected_results_prompted = {"ginis32": ginis32_p,  
                         "peaks32": peaks32_p,
                         "ginis64": ginis64_p, 
                         "peaks64": peaks64_p,
                         #"ginis128": ginis128_p,
                         #"peaks128": peaks128_p,
                         "knn": knns_p, 
                         "r2_aff": r2s_p}
    results_knn, results_bss, dfs = make_table(dataset, mteb_scores, collected_results)
    results_knn_p, results_bss_p, dfs_p = make_table(dataset, mteb_scores, collected_results_prompted)

    return results_knn, results_knn_p, results_bss, results_bss_p, dfs, dfs_p
    #return to_latex_rows_with_prompt(results_knn, results_knn_p, statistics_to_include=indicator)

In [88]:
rows={"knn": [], "gini32": [], "gini64": [], "gini128": []}
dataframes = {}
dataframes_p = {}
for d, d_ in dataset_map.items():
    print(d)
    results_knn, results_knn_p, results_bss, results_bss_p, dfs, dfs_p = runner(models, d,d_)
    dataframes[d] = dfs
    dataframes_p[d] = dfs_p
    #print(results_bss)
    rows["knn"].append(to_latex_rows_with_prompt(results_knn, results_knn_p, statistics_to_include="knn"))
    rows["gini32"].append(to_latex_rows_with_prompt(results_bss, results_bss_p, statistics_to_include="ginis32"))
    rows["gini64"].append(to_latex_rows_with_prompt(results_bss, results_bss_p, statistics_to_include="ginis64"))
    #rows["gini128"].append(to_latex_rows_with_prompt(results_bss, results_bss_p, statistics_to_include="gini128"))



Tatoeba
Reading MTEB results...
Reading BSS results...
Reading BSS results...
Reading NN-results...
Reading BSS results...
Reading BSS results...
Reading NN-results...


In [89]:
for r in rows["knn"]:
    print(r)

Tatoeba:deu-eng  & \textbf{0.853}$^{\ddagger}$ & 0.798$^{\ddagger}$ & 25\\
Tatoeba:bul-eng  & \textbf{0.964}$^{\ddagger}$ & \textbf{0.897}$^{\ddagger}$ & 25\\
Tatoeba:ukr-eng  & \textbf{0.922}$^{\ddagger}$ & \textbf{0.876}$^{\ddagger}$ & 25\\
Tatoeba:ita-eng  & \textbf{0.916}$^{\ddagger}$ & \textbf{0.852}$^{\ddagger}$ & 25\\
Tatoeba:fra-eng  & \textbf{0.878}$^{\ddagger}$ & \textbf{0.829}$^{\ddagger}$ & 25\\
Tatoeba:vie-eng  & \textbf{0.953}$^{\ddagger}$ & \textbf{0.870}$^{\ddagger}$ & 25\\
Tatoeba:ell-eng  & \textbf{0.962}$^{\ddagger}$ & \textbf{0.875}$^{\ddagger}$ & 25\\
Tatoeba:jpn-eng  & \textbf{0.952}$^{\ddagger}$ & \textbf{0.888}$^{\ddagger}$ & 25\\
Tatoeba:hin-eng  & \textbf{0.930}$^{\ddagger}$ & \textbf{0.810}$^{\ddagger}$ & 25\\
Tatoeba:fin-eng  & \textbf{0.975}$^{\ddagger}$ & \textbf{0.914}$^{\ddagger}$ & 25\\


In [90]:
for r in rows["gini64"]:
    print(r)
#print(rows.keys())

Tatoeba:deu-eng  & \textbf{0.838}$^{\ddagger}$ & \textbf{0.945}$^{\ddagger}$ & 25\\
Tatoeba:bul-eng  & \textbf{0.917}$^{\ddagger}$ & \textbf{0.895}$^{\ddagger}$ & 25\\
Tatoeba:ukr-eng  & \textbf{0.937}$^{\ddagger}$ & \textbf{0.935}$^{\ddagger}$ & 25\\
Tatoeba:ita-eng  & \textbf{0.920}$^{\ddagger}$ & \textbf{0.898}$^{\ddagger}$ & 25\\
Tatoeba:fra-eng  & \textbf{0.849}$^{\ddagger}$ & \textbf{0.879}$^{\ddagger}$ & 25\\
Tatoeba:vie-eng  & \textbf{0.874}$^{\ddagger}$ & \textbf{0.884}$^{\ddagger}$ & 25\\
Tatoeba:ell-eng  & \textbf{0.872}$^{\ddagger}$ & \textbf{0.853}$^{\ddagger}$ & 25\\
Tatoeba:jpn-eng  & \textbf{0.854}$^{\ddagger}$ & \textbf{0.844}$^{\ddagger}$ & 25\\
Tatoeba:hin-eng  & \textbf{0.892}$^{\ddagger}$ & \textbf{0.885}$^{\ddagger}$ & 25\\
Tatoeba:fin-eng  & \textbf{0.888}$^{\ddagger}$ & \textbf{0.868}$^{\ddagger}$ & 25\\


In [91]:
display(dataframes["Tatoeba"]["fin-eng"])

,model,mteb_score,ginis32,peaks32,ginis64,peaks64,knn,r2_aff
0,BAAI__bge-m3,0.961333,0.145993,2.313683,0.103790,1.598904,0.511527,0.787740
1,google__embeddinggemma-300m,0.248103,0.071929,1.189799,0.063610,1.177438,0.161619,0.390558
2,BAAI__bge-base-en-v1.5,0.031416,0.049277,1.560067,0.039428,1.089203,0.121587,0.362305
3,Alibaba-NLP__gte-multilingual-base,0.883033,0.136113,3.335228,0.107982,3.214796,0.344664,0.668939
4,minishlab__potion-multilingual-128M,0.399815,0.106017,1.884139,0.077760,1.247733,0.200209,0.319823
5,intfloat__e5-mistral-7b-instruct,0.811952,0.158718,3.375883,0.126064,3.294666,0.301689,0.942859
6,sentence-transformers__LaBSE,0.963667,0.164624,3.686991,0.121170,3.991422,0.532110,0.824198
7,Qwen__Qwen3-Embedding-8B,0.889219,0.159825,3.643630,0.118799,3.358466,0.334059,0.940505
8,Snowflake__snowflake-arctic-embed-l,0.017025,0.057143,1.519622,0.044395,1.346080,0.115075,0.400224
9,thenlper__gte-small,0.042096,0.042336,1.105374,0.043769,1.108249,0.136763,0.274026


In [92]:
df = dataframes["Tatoeba"]["fin-eng"]
fig = px.scatter(df, x="knn", y = "mteb_score", hover_data="model", template="none")#, text="model")

models_of_interest = {"sentence-transformers__LaBSE": (-1,-1), 
                      "BAAI__bge-m3": (-0.5, -0.5),
                      "BAAI__bge-base-en-v1.5":(-0.5, 0.5),
                      "intfloat__multilingual-e5-large":(0.7,-0.7), 
                      "Qwen__Qwen3-Embedding-8B":(1,-1),
                      "ibm-granite__granite-embedding-107m-multilingual":(-1,1),
                      "intfloat__multilingual-e5-small":(-0.7,0.7), 
                      "ibm-granite__granite-embedding-125m-english":(1,0.5),
}
for model_, i in models_of_interest.items():
    assert model_ in df["model"].tolist(), f"no {model_} found"
    fig.add_annotation(x = df[df.model==model_]["knn"].tolist()[0] , y = df[df.model==model_]["mteb_score"].tolist()[0], 
                        ay=i[1]*-80, ax=i[0]*80, text=model_.split("__")[-1], showarrow=True, arrowwidth=1, font_size=20)
#fig.add_annotation(x = 0.5 , y = 0.5, text= "BAAI_bge-m3", showarrow=True)
fig.update_yaxes(title="MTEB-score")#$\\bar{|\Delta|}$")
fig.update_xaxes(title="")
fig.update_layout(
    font_family="Times New Roman",
    title_font_family="Times New Roman",
)
fig.update_layout(margin=dict(l=40, r=10, t=20, b=20))
fig.update_yaxes(title_font_family="Times New Roman", title_font_size=20)
fig.update_layout(width=1200, height=400)
fig.show()
fig.write_image("knn_tatoeba_en-fi_annotated_figure.png", scale =3)

#top_fig.write_image("ICA32-tatoeba-en-fr-intfloat__multilingual-e5-large-instruct.png")

In [93]:
df = dataframes["Tatoeba"]["fin-eng"]
fig = px.scatter(df, x="ginis32", y = "mteb_score", hover_data="model", template="none")#, text="model")

models_of_interest = ["sentence-transformers__LaBSE", 
                      "BAAI__bge-base-en-v1.5",
                      "BAAI__bge-m3",
                      "intfloat__multilingual-e5-large", 
                      "ibm-granite__granite-embedding-107m-multilingual",
                      "Qwen__Qwen3-Embedding-8B",
                      "intfloat__multilingual-e5-small", 
                      "ibm-granite__granite-embedding-125m-english",
                      ]
for i, model_ in enumerate(models_of_interest):
    assert model_ in df["model"].tolist(), f"no {model_} found"
    fig.add_annotation(x = df[df.model==model_]["ginis32"].tolist()[0] , y = df[df.model==model_]["mteb_score"].tolist()[0], ay=((i)%2 -0.5)*40, ax=((i+1)%2 -0.5)*-60, text=model_.split("__")[-1], showarrow=True, arrowwidth=1)
#fig.add_annotation(x = 0.5 , y = 0.5, text= "BAAI_bge-m3", showarrow=True)

fig.update_yaxes(title="MTEB-score")#$\\bar{|\Delta|}$")
fig.update_xaxes(title="$Ret_k$")
fig.update_layout(width=1200)
fig.show()